# Lowest Common Ancestor (with parent pointers)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Hash Tables, Trees · **Difficulty/Frequency:** Uncommon (3/10)

> **Language note.** The official answer is Java; this notebook implements the same three algorithms in Python so every claim is executable. The Java reference is preserved verbatim in [`README.md`](README.md).

## Concepts

**What this problem is really testing:**
- Spotting that **parent pointers turn a tree problem into a linked-list problem**
- The **align-then-walk-together** technique for two paths of unequal length
- Trading O(h) space for O(1) by replacing a set with arithmetic

**First-principles primer — what is each piece?**

- **Ancestor.** Any node on the path from a node up to the root — including the node itself, by the usual convention (which is what makes "one node is the ancestor of the other" work cleanly).
- **Lowest common ancestor.** The **deepest** node that is an ancestor of both. "Lowest" means furthest from the root, i.e. the *last* place the two paths agree on the way down — equivalently, the *first* place they meet on the way up.
- **Parent pointer.** Each node knows its parent. This is the gift in this problem: it means you can walk **upward**, which an ordinary binary tree cannot do.
- **Height `h`.** The longest root-to-leaf path. All the complexities here are in terms of `h`, not `n` — O(log n) for a balanced tree, O(n) for a degenerate one.

**The reframe that solves it:**

> With parent pointers, the chain of ancestors from any node up to the root is a **linked list**. Both `p` and `q` have such a chain, and **both end at the same node — the root**.

Two linked lists that end at the same node form a **Y shape**. The LCA is exactly the point where they merge. This is literally the classic "find the intersection of two linked lists" problem wearing a tree costume — and recognising that is most of the answer.

**Why aligning by depth works.** The two chains have different lengths, so stepping both up together from the start compares nodes at *different depths* — meaningless. Chop the extra prefix off the longer chain and both pointers are the same distance from the root; from then on they are always at equal depth, so the **first** time they coincide is necessarily the **lowest** common ancestor.

**Simple worked example.**

```
        3
      /   \
     5     1
    / \   / \
   6   2 0   8
      / \
     7   4
```

`p = 7` (depth 4), `q = 1` (depth 2).

1. **Align:** `p` is 2 deeper, so climb it twice: `7 -> 2 -> 5`. Now both are at depth 2.
2. **Walk together:** `5 != 1` → climb both: `5 -> 3`, `1 -> 3`. Now `3 == 3`. **LCA = 3.** ✅

## Problem Statement

Given two nodes `p` and `q` in a binary tree where **every node has a `parent` pointer**, return their lowest common ancestor.

```java
class Node { int val; Node left, right, parent; }
```

By the usual convention a node is its own ancestor, so if `p` is an ancestor of `q`, the answer is `p`.

In [ ]:
from typing import Dict, List, Optional, Set, Tuple


class Node:
    __slots__ = ("val", "left", "right", "parent")

    def __init__(self, val: int) -> None:
        self.val = val
        self.left: Optional["Node"] = None
        self.right: Optional["Node"] = None
        self.parent: Optional["Node"] = None

    def __repr__(self) -> str:
        return f"Node({self.val})"


def build_tree(pairs: List[Tuple[int, Optional[int], Optional[int]]]) -> Dict[int, Node]:
    """pairs: (value, left value or None, right value or None). Wires parents automatically."""
    nodes: Dict[int, Node] = {}

    def get(v: int) -> Node:
        if v not in nodes:
            nodes[v] = Node(v)
        return nodes[v]

    for val, left, right in pairs:
        node = get(val)
        if left is not None:
            node.left = get(left)
            node.left.parent = node               # the parent pointer is the whole gift
        if right is not None:
            node.right = get(right)
            node.right.parent = node
    return nodes


# The tree from the worked example above.
TREE = build_tree([
    (3, 5, 1),
    (5, 6, 2),
    (2, 7, 4),
    (1, 0, 8),
])

### Approach 1 — Naive (walk up from the root for every comparison)

**Idea:** the version you would write with no insight at all — for each ancestor of `p` (found by walking up), ask "is this also an ancestor of `q`?" by walking `q`'s chain from scratch every time. Return the first hit, which is the deepest because we test `p`'s ancestors bottom-up.

Correct, and quadratic in the height for no reason: `q`'s chain is re-walked once per ancestor of `p`.

**Time complexity:** **O(h²)**.

**Space complexity:** O(1).

In [ ]:
def lca_naive(p: Node, q: Node) -> Optional[Node]:
    a = p
    while a is not None:                          # p's ancestors, deepest first
        b = q
        while b is not None:                      # ...re-walking q's chain EVERY time
            if a is b:
                return a
            b = b.parent
        a = a.parent
    return None                                   # different trees

### Approach 2 — Hash set of one chain

**Idea:** the obvious fix — walk up from `p` once, recording every ancestor in a set. Then walk up from `q` and return the first node already in the set.

This is the answer most people reach first, and it is genuinely good: linear, simple, and hard to get wrong. It also generalises to *k* nodes, and to graphs where "walk up" is not a single chain. Its only cost is the O(h) set.

**Time complexity:** **O(h)**.

**Space complexity:** **O(h)** — the set holds one entry per ancestor of `p`.

In [ ]:
def lca_hashset(p: Node, q: Node) -> Optional[Node]:
    seen: Set[int] = set()
    node: Optional[Node] = p
    while node is not None:                       # record p's entire ancestor chain
        seen.add(id(node))
        node = node.parent

    node = q
    while node is not None:                       # first of q's ancestors already seen
        if id(node) in seen:
            return node                           # walking UP means the first hit is the LOWEST
        node = node.parent
    return None

### Approach 3 — Optimal (align by depth, then walk together)

**Idea:** replace the set with arithmetic. Measure both depths, climb the deeper node by exactly the difference, then step both up in lockstep until they coincide.

**Why it is correct:** after aligning, the two pointers are always at the same depth. Any common ancestor sits at some depth `d`; the pointers reach depth `d` simultaneously, so the *first* coincidence is at the greatest depth — which is the definition of *lowest*.

**Why one node being the ancestor of the other needs no special case:** if `p` is an ancestor of `q`, then aligning climbs `q` up to exactly `p`'s depth, which lands it *on* `p`. The lockstep loop sees them equal immediately and returns. Worth tracing aloud, because interviewers ask.

**Time complexity:** **O(h)** — two depth walks plus one aligned climb.

**Space complexity:** **O(1)** — three integers and two pointers. This is the whole reason to prefer it over Approach 2.

In [ ]:
def depth(node: Optional[Node]) -> int:
    d = 0
    while node is not None:
        node = node.parent
        d += 1
    return d


def lca(p: Node, q: Node) -> Optional[Node]:
    dp, dq = depth(p), depth(q)

    while dp > dq:                                # chop the extra prefix off the deeper chain
        p = p.parent
        dp -= 1
    while dq > dp:
        q = q.parent
        dq -= 1

    while p is not q:                             # now always at equal depth: step in lockstep
        p = p.parent
        q = q.parent                              # both hit None together if they are different trees
    return p

### Follow-up — no parent pointers

**Idea:** the parent pointer is what made upward walking possible. Without it you must search **downward** from the root, and the shape of the answer changes completely.

The classic recursion returns, for each subtree, "did you find `p` or `q` below you?":

- If a node **is** `p` or `q`, report it upward.
- If **both** children report a find, this node is where the two paths diverge — it is the LCA.
- If only one child reports, pass that report up unchanged.

The first node that sees reports from *both* sides is necessarily the deepest such node, because any lower one would have been in a single subtree.

**Time complexity:** O(n) — every node may be visited.

**Space complexity:** O(h) for the recursion stack. Note the trade: without parent pointers you cannot achieve O(1) space, and you must visit the whole tree rather than just two root-paths.

In [ ]:
def lca_no_parent(root: Optional[Node], p: Node, q: Node) -> Optional[Node]:
    """Downward DFS. Assumes both p and q are present in the tree."""
    if root is None or root is p or root is q:
        return root                               # found one - report it upward
    left = lca_no_parent(root.left, p, q)
    right = lca_no_parent(root.right, p, q)
    if left is not None and right is not None:
        return root                               # p and q are on opposite sides => this is the LCA
    return left if left is not None else right    # only one side found something: pass it up

## Verification

Check the worked example, every structural edge case (self, ancestor-of, siblings, root), and then confirm all four implementations agree on **every pair** in randomly generated trees.

In [ ]:
import random

PAIR_IMPLS = [lca, lca_hashset, lca_naive]
root = TREE[3]

# --- The worked example ---
for fn in PAIR_IMPLS:
    assert fn(TREE[7], TREE[1]) is TREE[3], fn.__name__     # deep left vs shallow right
    assert fn(TREE[5], TREE[1]) is TREE[3], fn.__name__     # the two children of the root
    assert fn(TREE[6], TREE[4]) is TREE[5], fn.__name__     # both inside the left subtree
    assert fn(TREE[7], TREE[4]) is TREE[2], fn.__name__     # siblings
    assert fn(TREE[0], TREE[8]) is TREE[1], fn.__name__

# --- Edge cases ---
for fn in PAIR_IMPLS:
    assert fn(TREE[5], TREE[5]) is TREE[5], f"{fn.__name__}: a node is its own ancestor"
    assert fn(TREE[5], TREE[4]) is TREE[5], f"{fn.__name__}: p is an ancestor of q"
    assert fn(TREE[4], TREE[5]) is TREE[5], f"{fn.__name__}: ...and the reverse"
    assert fn(TREE[3], TREE[7]) is TREE[3], f"{fn.__name__}: the root is an ancestor of everything"
    assert fn(TREE[3], TREE[3]) is TREE[3], f"{fn.__name__}: root with itself"

# Two nodes in DIFFERENT trees have no common ancestor
other = build_tree([(100, 101, 102)])
for fn in PAIR_IMPLS:
    assert fn(TREE[7], other[101]) is None, f"{fn.__name__}: disjoint trees -> None"

# A single-node tree
lone = build_tree([(42, None, None)])
for fn in PAIR_IMPLS:
    assert fn(lone[42], lone[42]) is lone[42], fn.__name__

# A degenerate (linked-list shaped) tree - the O(h) = O(n) worst case
chain = build_tree([(i, i + 1, None) for i in range(50)])
for fn in PAIR_IMPLS:
    assert fn(chain[49], chain[10]) is chain[10], f"{fn.__name__}: ancestor in a skewed tree"
    assert fn(chain[0], chain[49]) is chain[0], fn.__name__

# --- depth() agrees with the actual distance to the root ---
assert depth(TREE[3]) == 1 and depth(TREE[5]) == 2 and depth(TREE[7]) == 4

# --- The no-parent-pointer version agrees with the rest ---
for a in TREE.values():
    for b in TREE.values():
        assert lca_no_parent(root, a, b) is lca(a, b), (a, b)


# --- Randomised trees: all four implementations must agree on every pair ---
def random_tree(n: int, rng: random.Random):
    """Build a random binary tree of n nodes; return (root, list of nodes)."""
    nodes = [Node(i) for i in range(n)]
    attachable = [nodes[0]]
    for i in range(1, n):
        parent = rng.choice(attachable)
        if parent.left is None and (parent.right is not None or rng.random() < 0.5):
            parent.left = nodes[i]
        elif parent.right is None:
            parent.right = nodes[i]
        else:
            parent.left = nodes[i]
        nodes[i].parent = parent
        if parent.left is not None and parent.right is not None:
            attachable.remove(parent)             # both slots filled
        attachable.append(nodes[i])
    return nodes[0], nodes


def brute_force_lca(a: Node, b: Node) -> Optional[Node]:
    """Independent reference: the deepest node present in both ancestor chains."""
    def chain(n):
        out = []
        while n is not None:
            out.append(n)
            n = n.parent
        return out

    ca, cb = chain(a), set(id(x) for x in chain(b))
    for node in ca:                               # deepest first
        if id(node) in cb:
            return node
    return None


rng = random.Random(43)
for _ in range(60):
    n = rng.randint(1, 30)
    r, nodes = random_tree(n, rng)
    for a in nodes:
        for b in nodes:
            expected = brute_force_lca(a, b)
            for fn in PAIR_IMPLS:
                assert fn(a, b) is expected, (fn.__name__, a.val, b.val)
            assert lca_no_parent(r, a, b) is expected, (a.val, b.val)

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Many queries on the same tree.** Answering each pair in O(h) is fine for a handful of queries and wasteful for millions. **Binary lifting** precomputes `up[k][v]` = the 2^k-th ancestor of `v` in O(n log n), after which the "climb the depth difference" step becomes O(log n) jumps instead of O(h) single steps — and the lockstep walk becomes a descending binary search over the same table. The **Euler tour + RMQ** approach goes further: O(n) or O(n log n) preprocessing for genuinely **O(1)** per query. The trade is always the same — preprocessing time and memory in exchange for per-query speed, and the break-even is roughly "more than h queries".
- **Nodes given by value rather than by reference.** You first have to *find* them, which is an O(n) traversal, and that dominates everything. Worth noting that this changes the problem's character: the parent-pointer trick optimises the part that is no longer the bottleneck. If values are unique, one pass building `value -> node` makes all later lookups O(1).
- **k-ary trees.** Completely unchanged. Nothing in Approach 3 mentions `left` or `right` — it only walks `parent`. The ancestor chain is a linked list regardless of how many children a node has, which is a nice illustration of how the reframing generalises where a downward DFS would need rewriting.
- **Duplicate values.** Everything here compares by **identity** (`is` / `id()`), not by value, which is what the problem's "given two nodes" phrasing demands. Comparing `node.val` would silently return the wrong ancestor whenever values repeat — a real bug in a tree of, say, employee records with duplicate names.
- **The linked-list connection.** This is exactly LeetCode 160, "Intersection of Two Linked Lists", with `parent` playing the role of `next`. The two well-known solutions map across directly: the hash-set approach (Approach 2) and the length-alignment approach (Approach 3). There is also the cute two-pointer variant — walk both pointers, and when one hits the end, restart it at the *other* list's head; both then travel `len(a) + len(b)` and meet at the intersection. It is O(1) space with no depth computation at all, and worth mentioning as a party trick, though the explicit alignment reads far more clearly.

## Empirical complexity check

Compare the **nested-walk** version (Approach 1, O(h²)) with **align-and-walk** (Approach 3, O(h)) on a deliberately degenerate tree — a single chain — so that `h = n` and the difference is unmistakable.

| Growth when the height doubles | What it means |
|---|---|
| ~4x | quadratic — `q`'s chain is re-walked once per ancestor of `p` |
| ~2x | linear — each chain is walked a constant number of times |

The hash-set version is included as a control: also O(h), but paying O(h) memory and hashing costs for what arithmetic does for free.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

QUERIES = 200


def make_chain(n):
    """A degenerate tree: one chain of n nodes, so height == n."""
    nodes = [Node(i) for i in range(n)]
    for i in range(1, n):
        nodes[i - 1].left = nodes[i]
        nodes[i].parent = nodes[i - 1]
    deep, shallow = nodes[-1], nodes[n // 2]
    return (deep, shallow)


def run_naive(deep, shallow):
    for _ in range(QUERIES):
        lca_naive(deep, shallow)          # O(h^2)


def run_hashset(deep, shallow):
    for _ in range(QUERIES):
        lca_hashset(deep, shallow)        # O(h) time, O(h) space


def run_optimal(deep, shallow):
    for _ in range(QUERIES):
        lca(deep, shallow)                # O(h) time, O(1) space


benchmark(
    {"Approach 1 - nested walk O(h^2)": run_naive,
     "Approach 2 - hash set O(h) time, O(h) space": run_hashset,
     "Approach 3 - align + walk O(h) time, O(1) space": run_optimal},
    make_chain,
    sizes=[100, 200, 400, 800],
    repeats=2,
)

## Patterns learned

- **A parent pointer turns a tree into a linked list.** The ancestor chain is a list, and two chains ending at the same root form a Y. Recognising that this *is* "intersection of two linked lists" is the whole insight — and it is why the k-ary case needs no changes at all.
- **Align, then walk together.** Whenever two sequences must be compared position-by-position but have different lengths, measure both, skip the difference, then step in lockstep. Same technique in list intersection, diffing, and merge-style joins.
- **Arithmetic can replace a hash set.** Approach 2 stores O(h) nodes to answer "have I seen this?". Approach 3 answers the same question with two integers, because *depth* already encodes the position. Look for a computable invariant before reaching for memory.
- **Walking upward, the first hit is the lowest.** Direction of traversal decides which extreme you find. Going up returns the deepest common ancestor for free; going down would need extra bookkeeping.
- **State complexity in terms of the right variable.** O(h), not O(n) — then add that h is log n balanced, n skewed. The benchmark here deliberately uses a degenerate chain to make h = n visible.
- **Compare by identity, not by value.** `is` / `id()`, because "two nodes" means two objects. Value comparison breaks the moment values repeat.
- **Preprocess only when queries are many.** One query: O(h) and no memory. Millions of queries on a fixed tree: binary lifting or Euler tour + RMQ. Knowing the break-even is the answer, not knowing the fancier algorithm.